# Customer Data Cleaning & Preprocessing with Python

**Portfolio practice project** using a fictional customer dataset.

This notebook demonstrates data inspection, text standardization, duplicate handling, missing-value treatment, validation, and exporting cleaned data.

## 1. Import libraries

Pandas is used to work with tabular data. NumPy provides `np.nan` for marking invalid values as missing.

In [ ]:
import pandas as pd
import numpy as np

## 2. Load the original dataset

Keep the original data unchanged and make a working copy. Update the file path if your CSV is stored in another folder.

In [ ]:
original_df = pd.read_csv("raw_customers.csv")
df = original_df.copy()

print("Original shape:", df.shape)
display(df.head())

## 3. Inspect data quality

Check column types, missing values, and exact duplicate rows. A full-row duplicate check does not detect records that match only on a selected field such as Email.

In [ ]:
df.info()
print("\nMissing values by column:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nRepeated email values:", df.duplicated(subset=["Email"], keep="first").sum())

## 4. Standardize text fields

Remove leading/trailing spaces and use consistent title case. The `Lhr` to `Lahore` mapping is applied because the intended city abbreviation is known in this practice dataset.

In [ ]:
df["Name"] = df["Name"].str.strip().str.title()
df["City"] = df["City"].str.strip().str.title()
df["City"] = df["City"].replace({"Lhr": "Lahore"})

print("Unique cities:", sorted(df["City"].dropna().unique()))

## 5. Handle the repeated customer record

For this exercise, the two Ali Khan rows have the same email and matching customer details, so we treat the second row as a duplicate and retain the first. In a real client project, confirm the business rule and investigate conflicting records before removing anything.

In [ ]:
print("Rows before duplicate handling:", len(df))
df = df.drop_duplicates(subset=["Email"], keep="first")
print("Rows after duplicate handling:", len(df))

## 6. Handle invalid and missing ages

Age 150 is treated as invalid under the provisional rule for this fictional exercise. It is changed to missing. We then calculate the median from the remaining unique customer records and use it to fill the two missing age entries. The imputed age is an estimate, not a verified value.

In [ ]:
df.loc[df["Age"] > 100, "Age"] = np.nan
median_age = df["Age"].median()
print("Median age used for imputation:", median_age)
df["Age"] = df["Age"].fillna(median_age)

## 7. Validate the cleaned dataset

We preserve the missing email and purchase amount because their correct values are unknown. We do not replace a missing purchase amount with zero.

In [ ]:
df = df.reset_index(drop=True)

print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nRepeated emails:", df.duplicated(subset=["Email"]).sum())
print("Age range:", df["Age"].min(), "to", df["Age"].max())
print("Rows with age above 100:", len(df[df["Age"] > 100]))
print("\nCleaned preview:")
display(df)

## 8. Export the cleaned files

The CSV is convenient for data exchange. The Excel file is convenient for clients who prefer spreadsheets.

In [ ]:
df.to_csv("cleaned_customers.csv", index=False)
df.to_excel("cleaned_customers.xlsx", index=False)
print("Export complete.")

## 9. Project notes

- Original records: 10
- Final records: 9
- Duplicate records removed: 1
- Median age used: 27
- Missing email and purchase amount values retained
- Age 150 treated as invalid under an exercise-specific rule

**Real-world note:** Always confirm validation rules and duplicate-handling decisions with the data owner. Document any imputation and avoid guessing missing business data.